In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [11]:
def load_words(path="names.txt"):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().splitlines()

w = load_words()
words = w[:30]

In [12]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(stoi)

In [13]:
block_size = 8

def build_dataset(words):
    X, Y = [], []

    for w in words:
        chs = ['.'] + list(w) + ['.']
        for i in range(len(chs)-1):
            context = chs[:i+1]
            target = chs[i+1]

            context = context[-block_size:]
            context = ['.'] * (block_size - len(context)) + context

            X.append([stoi[c] for c in context])
            Y.append(stoi[target])

    return torch.tensor(X), torch.tensor(Y)

X, Y = build_dataset(words)

In [14]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=64, n_head=4, n_layer=2):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=n_head,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layer)

        self.ln = nn.LayerNorm(n_embd)
        self.fc = nn.Linear(n_embd, vocab_size)

        self.block_size = block_size

    def forward(self, x):
        B, T = x.shape

        tok_emb = self.token_embedding(x)
        pos = torch.arange(T)
        pos_emb = self.position_embedding(pos)

        x = tok_emb + pos_emb

        # masque causal (important !)
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
        x = self.transformer(x, mask=mask)

        x = self.ln(x)
        logits = self.fc(x)

        return logits

In [15]:
model = MiniGPT(vocab_size, block_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

In [16]:
for step in range(3000):
    logits = model(X)

    logits_last = logits[:, -1, :]
    loss = F.cross_entropy(logits_last, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 300 == 0:
        print(loss.item())

3.2356832027435303
0.5695686340332031
0.534856379032135
0.5419516563415527
0.5412797927856445
0.5322221517562866
0.5232266187667847
0.5383663773536682
0.515024721622467
0.5162185430526733


In [17]:
@torch.no_grad()
def generate_transformer(model):
    out = []
    context = [0] * block_size

    while True:
        x = torch.tensor([context])
        logits = model(x)

        probs = F.softmax(logits[:, -1, :], dim=1)
        ix = torch.multinomial(probs, 1).item()

        if ix == 0:
            break

        out.append(itos[ix])
        context = context[1:] + [ix]

    return ''.join(out)


for _ in range(20):
    print(generate_transformer(model))

isabella
layla
victoria
elizabeth
elizabeth
elizabeth
avery
elizabeth
elizabeth
scarlett
avery
sophia
harper
layla
victoria
chloe
aria
elizabeth
zoey
nora
